# K-Nearest Neighbors Regression

## 1. Introduction
Predicts from nearby training examples, averaging targets using a chosen neighborhood and distance weighting.

**Why / when:** Use the model when its assumptions and data geometry fit the task; compare it to a simple baseline. **Hindi:** Is model ka chunav data aur business problem ke hisaab se karein; sirf popularity ke basis par nahin.

**Business example:** Local interpolation on small datasets with meaningful distance geometry.

This lesson uses deterministic synthetic data so it runs offline. Replace it with a licensed, documented dataset after checking its schema and provenance.

## 2. Mathematical intuition
ŷ(x)=Σᵢwᵢyᵢ/Σᵢwᵢ over the k nearest points.

The target is a real number. Let `n` be observations, `p` features, `X` the design matrix, `y` the target, and `f̂` the learned prediction function. Optimize only on training data.

## 3. Working mechanism
1. Learn parameters or partitions from training examples.
2. Predict outcomes for unseen rows.
3. Measure error on a holdout that was not used for fitting or tuning.

**Small numeric example:** For two neighbors with targets 10 and 14 at equal distance, KNN predicts (10 + 14) / 2 = 12. For a tree leaf with targets 10 and 14, the squared-error-optimal constant is also 12. Scale features, tune k and distance weighting, and account for inference/memory cost and the curse of dimensionality.

## 4. Algorithm from scratch (NumPy)
The following self-contained least-squares reference fit makes the matrix mechanics concrete. It is a transparent educational baseline; the sklearn model below is the algorithm under study. Complexity for dense least squares is approximately O(n·p² + p³) time and O(n·p + p²) memory.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
X_scratch = np.array([[1., 0.], [1., 1.], [1., 2.], [1., 3.]])
y_scratch = np.array([1., 3., 5., 7.])
weights, *_ = np.linalg.lstsq(X_scratch, y_scratch, rcond=None)
print("fitted intercept and slope:", weights)
print("manual check at x=4:", weights[0] + weights[1] * 4)

### Scratch dry run
With points (0,1), (1,3), (2,5), (3,7), the fitted intercept is 1 and slope is 2. The reference fit illustrates least squares; algorithm-specific implementation notes: A NumPy scratch implementation computes Euclidean distances and averages the nearest targets.

## 5. Dataset and business context
We generate a deterministic offline dataset with numeric measurements and a continuous outcome. This is a pedagogical substitute, not a public benchmark or a claim about a real population. Features are `size`, `rooms`, `age`, and `distance`; target is a synthetic value influenced by them and noise. In production, document collection, consent, units, and target definition.

In [ ]:
import pandas as pd
rng = np.random.default_rng(42)
n = 600
data = pd.DataFrame({
    "size": rng.uniform(30, 250, n),
    "rooms": rng.integers(1, 8, n),
    "age": rng.uniform(0, 80, n),
    "distance": rng.uniform(0.2, 30, n),
})
data["value"] = 1200 * data["size"] + 9000 * data["rooms"] - 350 * data["age"] - 1800 * data["distance"] + rng.normal(0, 18000, n)
X = data.drop(columns="value")
y = data["value"]
print("shape:", data.shape)
display(data.head())

## 6. Exploratory data analysis
Inspect schema, nulls, duplicates, summary statistics, distributions, correlations, potential outliers, and the target before modeling. Charts help find data issues; they do not prove causation.

In [ ]:
print(data.dtypes)
print("missing values:\n", data.isna().sum())
print("duplicate rows:", data.duplicated().sum())
display(data.describe().T)
display(data.corr(numeric_only=True).round(2))
data.hist(figsize=(10, 7), bins=25)

## 7. Preprocessing without leakage
Split first. Fit imputers/scalers on training data only by placing them inside a Pipeline. The synthetic data has numeric features; a ColumnTransformer is included so categorical columns can be added safely.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
numeric = X.select_dtypes(include="number").columns
categorical = X.select_dtypes(exclude="number").columns
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])

## 8. Baseline model
Use a mean predictor as the minimum regression baseline. A useful model should beat it on held-out data.

In [ ]:
from sklearn.dummy import DummyRegressor
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
print("baseline holdout RMSE:", np.sqrt(np.mean((y_test - baseline.predict(X_test)) ** 2)))

## 9. Detailed training
The configured estimator is wrapped with preprocessing in one pipeline. This keeps transformations consistent during cross-validation, inference, and serialization.

In [ ]:
from sklearn.pipeline import Pipeline
from model import build_model
model = Pipeline([("preprocessor", preprocessor), ("model", build_model())])
model.fit(X_train, y_train)
print("estimator:", model.named_steps["model"])
print("train/test rows:", len(X_train), len(X_test))

## 10. Evaluation
Report complementary error metrics, compare train/test scores, inspect residuals, and avoid using test data for model selection. MAPE is unstable when true targets are zero or near zero; R² can be negative on test data.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)
def regression_metrics(actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {"MAE": mean_absolute_error(actual, predicted), "MSE": mse, "RMSE": np.sqrt(mse), "R2": r2_score(actual, predicted)}
print("train:", regression_metrics(y_train, pred_train))
print("test:", regression_metrics(y_test, pred_test))
residuals = y_test - pred_test
pd.Series(residuals, name="residual").hist(bins=25)

## 11. Overfitting and underfitting
A large train/test gap suggests overfitting; poor scores on both suggest underfitting or a data/feature problem. Use learning curves and validation curves on training folds, then regularize or simplify where appropriate.

In [ ]:
from sklearn.model_selection import learning_curve
sizes, train_scores, valid_scores = learning_curve(model, X_train, y_train, cv=3, scoring="neg_root_mean_squared_error", train_sizes=np.linspace(.25, 1, 4), n_jobs=1)
print("train RMSE by size:", -train_scores.mean(axis=1).round(1))
print("validation RMSE by size:", -valid_scores.mean(axis=1).round(1))

## 12. Hyperparameter optimization
Use cross-validation only on the training partition. GridSearchCV is exhaustive over a small grid; RandomizedSearchCV samples broader spaces efficiently. Bayesian/Optuna search is an optional extension, not required for this baseline.

In [ ]:
from sklearn.model_selection import GridSearchCV
search = GridSearchCV(model, {'model__n_neighbors': [3, 7]}, cv=3, scoring="neg_root_mean_squared_error", n_jobs=1)
search.fit(X_train, y_train)
print("best parameters:", search.best_params_)
print("best cross-validation RMSE:", -search.best_score_)

## 13. Feature importance and explainability
Permutation importance measures the holdout-score drop after shuffling one feature; it can be misleading with correlated inputs. SHAP and LIME are optional libraries; explanations describe model behavior, not causality. Check subgroup error and fairness before consequential use.

In [ ]:
from sklearn.inspection import permutation_importance
importance = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42, scoring="neg_mean_squared_error")
print(pd.Series(importance.importances_mean, index=X.columns).sort_values(ascending=False))

## 14. Advanced techniques
Compare robust losses, feature selection, ensembles, and model-specific regularization with nested or carefully separated validation. For skewed targets consider transformations only when their inverse and metric interpretation are handled correctly. Track compute and latency.

In [ ]:
# Advanced technique: model-agnostic conformal prediction intervals
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import train_test_split

# Reserve part of training data for calibration; test data remains untouched until evaluation.
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)
calibration_model = clone(model)
calibration_model.fit(X_fit, y_fit)
calibration_errors = np.abs(y_cal.to_numpy() - calibration_model.predict(X_cal))

alpha = 0.10  # target approximately 90% marginal coverage
n_cal = len(calibration_errors)
quantile_index = min(int(np.ceil((n_cal + 1) * (1 - alpha))) - 1, n_cal - 1)
interval_radius = np.sort(calibration_errors)[quantile_index]
test_predictions = calibration_model.predict(X_test)
lower_bounds = test_predictions - interval_radius
upper_bounds = test_predictions + interval_radius
test_coverage = np.mean((y_test.to_numpy() >= lower_bounds) & (y_test.to_numpy() <= upper_bounds))

print(f"calibration interval radius: {interval_radius:.3f}")
print(f"test interval coverage: {test_coverage:.1%}")
print(f"mean interval width: {2 * interval_radius:.3f}")
print("first five prediction intervals:")
for prediction, lower, upper in zip(test_predictions[:5], lower_bounds[:5], upper_bounds[:5]):
    print(f"prediction={prediction:.3f}, interval=[{lower:.3f}, {upper:.3f}]")


## 15. Production implementation
The sibling `train.py`, `predict.py`, `evaluate.py`, `preprocessing.py`, and `model.py` provide a config-driven, logged, seeded CLI workflow with schema checks and a serialized pipeline. Unit tests check fit/predict behavior. Validate artifacts and dependency versions before deployment.

In [ ]:
# Production implementation: package, reload, and smoke-test the fitted pipeline
import json
import pickle
from pathlib import Path

import sklearn
from sklearn.metrics import mean_squared_error

artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / "linear_regression_model.pkl"
feature_path = artifact_dir / "feature_columns.json"
metadata_path = artifact_dir / "model_metadata.json"

# Persist the complete fitted pipeline, including its preprocessing steps.
with model_path.open("wb") as f:
    pickle.dump(model, f)

feature_columns = list(X_train.columns)
feature_path.write_text(json.dumps(feature_columns, indent=2), encoding="utf-8")
test_rmse = float(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))
metadata = {
    "model_class": type(model.named_steps["model"]).__name__ if hasattr(model, "named_steps") else type(model).__name__,
    "sklearn_version": sklearn.__version__,
    "training_rows": len(X_train),
    "feature_count": len(feature_columns),
    "test_rmse": test_rmse,
    "artifact": str(model_path),
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

# Verify the saved artifact can be loaded and used before deployment.
with model_path.open("rb") as f:
    packaged_model = pickle.load(f)
smoke_prediction = float(packaged_model.predict(X_test.loc[:, feature_columns].head(1))[0])
print(f"Saved model: {model_path}")
print(f"Metadata: {metadata}")
print(f"Smoke-test prediction: {smoke_prediction:.3f}")


## 16. Deployment
A production REST service should use request/response schemas, health and readiness endpoints, bounded batch sizes, authentication/rate limits at the gateway, and safe model loading. Docker and a minimal Kubernetes manifest are included as starting examples; deployment requires environment-specific security review.

In [ ]:
# Production deployment example using a pickled model
import json
import pickle
from pathlib import Path

import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

MODEL_PATH = Path("artifacts/linear_regression_model.pkl")
FEATURES_PATH = Path("artifacts/feature_columns.json")

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

with open(FEATURES_PATH, "r", encoding="utf-8") as f:
    feature_columns = json.load(f)

app = FastAPI(title="Regression API")

class PredictRequest(BaseModel):
    features: dict

class PredictResponse(BaseModel):
    prediction: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
def predict(payload: PredictRequest):
    row = pd.DataFrame([payload.features], columns=feature_columns)
    row = row[feature_columns]
    pred = float(model.predict(row)[0])
    return {"prediction": pred}

print("FastAPI app ready for deployment")


## 17. MLOps
Log parameters, data/model versions, and metrics; monitor feature and prediction drift as well as delayed-label performance. Define alert thresholds, retraining approval, canary rollout, and rollback to a known-good artifact. Data versioning and external tracking are deployment integrations.

In [ ]:
# MLOps: save pickle artifact and track metadata
import json
import pickle
from datetime import datetime
from pathlib import Path

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / "linear_regression_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f)

feature_columns = list(X.columns)
with open(artifact_dir / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_columns, f)

run = {
    "timestamp": datetime.utcnow().isoformat(),
    "model_name": type(model).__name__,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "feature_count": X_train.shape[1],
    "rmse": round(mean_squared_error(y_test, model.predict(X_test), squared=False), 4),
    "artifact_path": str(model_path),
}

(artifact_dir / "model_metadata.json").write_text(json.dumps(run, indent=2))
print(run)


## 18. Performance optimization
Measure rather than assume: benchmark batch and single-row latency, throughput, memory, and artifact size on representative hardware. Vectorize preprocessing, bound concurrency, and choose model complexity based on service-level objectives.

In [ ]:
# Performance benchmarking: latency and memory
import time
import tracemalloc

import numpy as np

# Warm-up for stable timing
_ = model.predict(X_test[:10])

sample_size = min(500, len(X_test))
batch = X_test[:sample_size]

single_times = []
for _ in range(20):
    t0 = time.perf_counter()
    model.predict(X_test[:1])
    single_times.append(time.perf_counter() - t0)

t0 = time.perf_counter()
batch_preds = model.predict(batch)
batch_time = time.perf_counter() - t0

tracemalloc.start()
_ = model.predict(batch)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"single-row latency avg: {np.mean(single_times) * 1000:.3f} ms")
print(f"batch latency for {sample_size} rows: {batch_time:.4f} s")
print(f"batch throughput: {sample_size / batch_time:.2f} rows/s")
print(f"peak traced memory: {peak / 1024:.2f} KB")
print(f"prediction sample: {batch_preds[:5]}")


## 19. Production case study
**Problem:** Local interpolation on small datasets with meaningful distance geometry. Establish a point-in-time-correct data snapshot, train with a temporal or random split appropriate to collection, register a reproducible artifact, and expose batch/online inference. Monitor data quality, drift, prediction errors, latency, and business outcomes. Failure modes include upstream schema changes, delayed labels, and out-of-range inputs; define fail-closed validation and rollback.

## 20. Interview preparation
1. What does the model optimize? 2. What assumptions matter? 3. How does scaling affect it? 4. What is its bias/variance profile? 5. How do you detect leakage? 6. Why use a pipeline? 7. How do you tune it? 8. How do you interpret residuals? 9. What causes train/test gaps? 10. How would you monitor drift? 11. How would you handle missing values? 12. What is a useful baseline? 13. How do correlated features affect explanations? 14. How do you deploy safely? 15. How do you roll back? 16. What changes for large data? 17. When is a metric misleading? 18. How should the split reflect production? 19. What is the latency budget? 20. How do you communicate uncertainty? Answers: define the objective and assumptions, fit all transforms inside train-only folds, compare against simple baselines, select with validation, reserve test for final evaluation, monitor input and outcome quality, and roll out/rollback using versioned artifacts.

## 21. Exercises
**Beginner:** change the noise level and compare RMSE. **Intermediate:** add one categorical feature and preprocess it. **Advanced:** tune two parameters using cross-validation and report the untouched test score. **Production:** validate missing/extra columns and create a rollback plan. **Solution guidance:** preserve the test split, put every learned transformation in the pipeline, and record the random seed and configuration.